In [18]:
import utils, datetime, sagemaker, boto3, importlib
import pandas as pd
import numpy as np
import paths as p
from sagemaker.model_monitor.dataset_format import DatasetFormat
importlib.reload(utils)
pd.set_option('display.max_colwidth', None)  # show full text in each cell
pd.set_option('display.max_columns', None)   # show all columns
pd.set_option('display.width', None)         # don't wrap


In [3]:
region = 'us-east-1'
role = utils.get_sm_service_role_arn()

data_bucket='omm-test-bucket'
project_path = 'models/abalone'

model_package_group_name='abalone'
model_version=1
target_name='rings'
prediction_name=target_name+'_prediction'
endpoint_name='abalone-endpoint'
model_name = f'{model_package_group_name}-{str(model_version)}'

data_bucket='omm-test-bucket'
project_path = 'models/abalone'
feature_columns = ["length", "diameter", "height", "whole_weight", "shucked_weight", "viscera_weight", "shell_weight", "sex_I", "sex_M", "sex_F"]

boto_session=boto3.Session(region_name=region)
sm_client = boto_session.client('sagemaker', region_name=region)
sagemaker_session = sagemaker.Session(boto_session=boto_session)


## Deploy

In [9]:
model_name, model_arn = utils.create_model_object_from_registry(boto_session, model_package_group_name, model_version)
print(model_name +" "+ model_arn)


using existing model
abalone-1 arn:aws:sagemaker:us-east-1:088461143167:model/abalone-1


In [10]:
utils.deploy_model_endpoint(boto_session, model_name, endpoint_name, p.data_capture_dir, instance_type='ml.m5.large')

creating endpoint config
arn:aws:sagemaker:us-east-1:088461143167:endpoint-config/abalone-endpoint-config
updating endpoint
{'EndpointArn': 'arn:aws:sagemaker:us-east-1:088461143167:endpoint/abalone-endpoint', 'ResponseMetadata': {'RequestId': 'ce5d6cb4-442b-4696-81e2-cdc9fca7019f', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'ce5d6cb4-442b-4696-81e2-cdc9fca7019f', 'strict-transport-security': 'max-age=47304000; includeSubDomains', 'x-frame-options': 'DENY', 'content-security-policy': "frame-ancestors 'none'", 'cache-control': 'no-cache, no-store, must-revalidate', 'x-content-type-options': 'nosniff', 'content-type': 'application/x-amz-json-1.1', 'content-length': '84', 'date': 'Thu, 21 May 2026 17:43:45 GMT'}, 'RetryAttempts': 0}}


## Monitoring

<table class="min-w-full border-collapse text-sm leading-[1.7] whitespace-normal"><thead class="text-left"><tr><th scope="col" class="text-text-100 border-b-0.5 border-border-300/60 py-2 pr-4 align-top font-bold">Class</th><th scope="col" class="text-text-100 border-b-0.5 border-border-300/60 py-2 pr-4 align-top font-bold">Monitors</th></tr></thead><tbody><tr><td class="border-b-0.5 border-border-300/30 py-2 pr-4 align-top"><code class="bg-text-200/5 border border-0.5 border-border-300 text-danger-000 whitespace-pre-wrap rounded-[0.4rem] px-1 py-px text-[0.9rem]">DefaultModelMonitor</code></td><td class="border-b-0.5 border-border-300/30 py-2 pr-4 align-top">Data quality / data drift</td></tr><tr><td class="border-b-0.5 border-border-300/30 py-2 pr-4 align-top"><code class="bg-text-200/5 border border-0.5 border-border-300 text-danger-000 whitespace-pre-wrap rounded-[0.4rem] px-1 py-px text-[0.9rem]">ModelQualityMonitor</code></td><td class="border-b-0.5 border-border-300/30 py-2 pr-4 align-top">Model accuracy / prediction quality</td></tr><tr><td class="border-b-0.5 border-border-300/30 py-2 pr-4 align-top"><code class="bg-text-200/5 border border-0.5 border-border-300 text-danger-000 whitespace-pre-wrap rounded-[0.4rem] px-1 py-px text-[0.9rem]">ModelExplainabilityMonitor</code></td><td class="border-b-0.5 border-border-300/30 py-2 pr-4 align-top">SHAP feature importance drift</td></tr><tr><td class="border-b-0.5 border-border-300/30 py-2 pr-4 align-top"><code class="bg-text-200/5 border border-0.5 border-border-300 text-danger-000 whitespace-pre-wrap rounded-[0.4rem] px-1 py-px text-[0.9rem]">ModelBiasMonitor</code></td><td class="border-b-0.5 border-border-300/30 py-2 pr-4 align-top">Bias drift against protected groups</td></tr></tbody></table>

In [36]:
# Delete old monitors
schedules = sm_client.list_monitoring_schedules()
for s in schedules['MonitoringScheduleSummaries']:
    print(s['MonitoringScheduleName'], s['MonitoringScheduleStatus'])

sm_client.delete_monitoring_schedule(MonitoringScheduleName='abalone-model-bias-monitor')
sm_client.delete_monitoring_schedule(MonitoringScheduleName='abalone-model-quality-monitor')
sm_client.delete_monitoring_schedule(MonitoringScheduleName='abalone-data-quality-monitor')

abalone-model-bias-monitor Scheduled
abalone-model-quality-monitor Scheduled
abalone-data-quality-monitor Scheduled


{'ResponseMetadata': {'RequestId': '5e99154f-217e-440a-a1c4-215283860a1a',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '5e99154f-217e-440a-a1c4-215283860a1a',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'date': 'Thu, 21 May 2026 18:31:14 GMT',
   'content-length': '0'},
  'RetryAttempts': 2}}

In [28]:
baseline_full

,rings_prediction,rings,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,sex_F,sex_I,sex_M
0,8.144435,7,0.410,0.300,0.100,0.2820,0.1255,0.0570,0.0875,0,1,0
1,12.470012,13,0.450,0.360,0.125,0.4995,0.2035,0.1000,0.1700,1,0,0
2,9.323292,11,0.490,0.380,0.135,0.5415,0.2175,0.0950,0.1900,0,0,1
3,10.996564,14,0.430,0.340,0.125,0.3840,0.1375,0.0610,0.1460,0,1,0
4,10.479134,11,0.545,0.450,0.150,0.8795,0.3870,0.1500,0.2625,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
739,10.179646,10,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.1550,0,0,1
740,18.457285,20,0.675,0.545,0.195,1.7345,0.6845,0.3695,0.6050,1,0,0
741,9.217548,8,0.625,0.480,0.185,1.2065,0.5870,0.2900,0.2860,0,0,1
742,7.406839,5,0.400,0.290,0.100,0.2675,0.1205,0.0605,0.0765,0,1,0


In [43]:
baseline=pd.read_csv(p.baseline_file, header=0)
baseline_pred=pd.read_csv(p.baseline_pred_file, header=None)
baseline_pred.columns=[prediction_name]
baseline_full = pd.concat([baseline_pred, baseline], axis=1)
baseline_full[target_name] = baseline_full[target_name].astype(float)
baseline_full[prediction_name] = baseline_full[prediction_name].astype(float)


# Data Quality    → input features only
#                   detects: input distribution drift
baseline_full.drop(columns=[target_name, prediction_name]).to_csv(f'{p.dq_monitor_dir}/baseline.csv', index=False, header=True)

# Model Quality   → predictions + ground truth labels
#                   detects: accuracy degradation over time
baseline_full[[target_name, prediction_name]].to_csv(f'{p.mq_monitor_dir}/baseline.csv', index=False, header=True)

# Model Bias      → features + predictions + labels
#                   detects: unfair predictions against subgroups
baseline_full.to_csv(f'{p.mb_monitor_dir}/baseline.csv', index=False, header=True)

# Model Explainability → input features + predictions
#                        detects: feature importance drift
#                        uses SHAP values
baseline_full.drop(columns=[target_name]).to_csv(f'{p.me_monitor_dir}/baseline.csv', index=False, header=True)

### Data Quality

In [32]:
pd.read_csv(f'{p.dq_monitor_dir}/baseline.csv', header=0)

,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,sex_F,sex_I,sex_M
0,0.410,0.300,0.100,0.2820,0.1255,0.0570,0.0875,0,1,0
1,0.450,0.360,0.125,0.4995,0.2035,0.1000,0.1700,1,0,0
2,0.490,0.380,0.135,0.5415,0.2175,0.0950,0.1900,0,0,1
3,0.430,0.340,0.125,0.3840,0.1375,0.0610,0.1460,0,1,0
4,0.545,0.450,0.150,0.8795,0.3870,0.1500,0.2625,0,0,1
...,...,...,...,...,...,...,...,...,...,...
739,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.1550,0,0,1
740,0.675,0.545,0.195,1.7345,0.6845,0.3695,0.6050,1,0,0
741,0.625,0.480,0.185,1.2065,0.5870,0.2900,0.2860,0,0,1
742,0.400,0.290,0.100,0.2675,0.1205,0.0605,0.0765,0,1,0


In [33]:
data_quality_monitor = sagemaker.model_monitor.DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sagemaker_session
)

data_quality_monitor.suggest_baseline(
    baseline_dataset=f'{p.dq_monitor_dir}/baseline.csv',
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"{p.dq_monitor_dir}/info",
    wait=True,
    logs=False
)

INFO:sagemaker.image_uris:Defaulting to the only supported framework/algorithm version: .
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-05-21-18-24-32-312


...........................................................!

In [37]:
data_quality_monitor.create_monitoring_schedule(
    monitor_schedule_name='abalone-data-quality-monitor',
    endpoint_input='abalone-endpoint',
    output_s3_uri=f'{p.dq_monitor_dir}/reports',
    statistics=data_quality_monitor.baseline_statistics(),
    constraints=data_quality_monitor.suggested_constraints(),
    schedule_cron_expression=sagemaker.model_monitor.CronExpressionGenerator.hourly()
)
# transform job has no data capture but instead has separate files for inputs and ouputs
# monitor.create_monitoring_schedule(
#     monitor_schedule_name='abalone-batch-monitor',
#     batch_transform_input=sagemaker.model_monitor.BatchTransformInput(
#         data_captured_destination_s3_uri=f's3://{data_bucket}/batch-output/', # predictions
#         dataset_format=DatasetFormat.csv(header=False),
#         local_path='/opt/ml/processing/input' # also need to point at input features separately
#     ),
#     output_s3_uri=f's3://{data_bucket}/model-monitor/reports',
#     statistics=monitor.baseline_statistics(),
#     constraints=monitor.suggested_constraints(),
#     schedule_cron_expression=sagemaker.model_monitor.CronExpressionGenerator.hourly()
# )

INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: abalone-data-quality-monitor


### Model Quality

In [38]:
model_quality_monitor = sagemaker.model_monitor.ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sagemaker_session
)

model_quality_monitor.suggest_baseline(
    baseline_dataset=f'{p.mq_monitor_dir}/baseline.csv',
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f'{p.mq_monitor_dir}/info',
    problem_type='Regression',
    inference_attribute=prediction_name,   # target column header (named by PySpark)
    ground_truth_attribute=target_name, # output column header (only 1 output)
    wait=True,
    logs=False
)

INFO:sagemaker.image_uris:Defaulting to the only supported framework/algorithm version: .
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-05-21-18-31-34-226


...........................................................!

In [39]:
model_quality_monitor.create_monitoring_schedule(
    monitor_schedule_name='abalone-model-quality-monitor',
    endpoint_input=sagemaker.model_monitor.EndpointInput(
        endpoint_name=endpoint_name,
        destination='/opt/ml/processing/input/endpoint',
        inference_attribute=prediction_name  # column index of prediction in output
    ),
    ground_truth_input=p.ground_truth_dir,
    problem_type='Regression',
    output_s3_uri=f'{p.mq_monitor_dir}/reports',
    constraints=model_quality_monitor.suggested_constraints(),
    schedule_cron_expression=sagemaker.model_monitor.CronExpressionGenerator.hourly()
)

INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: abalone-model-quality-monitor


### Model Bias

In [42]:
pd.read_csv(f'{p.mb_monitor_dir}/baseline.csv').dtypes

rings_prediction    float64
rings                 int64
length              float64
diameter            float64
height              float64
whole_weight        float64
shucked_weight      float64
viscera_weight      float64
shell_weight        float64
sex_F                 int64
sex_I                 int64
sex_M                 int64
dtype: object

In [44]:
endpoint_config_name = sm_client.describe_endpoint(EndpointName=endpoint_name)['EndpointConfigName']
model_name = sm_client.describe_endpoint_config(EndpointConfigName=endpoint_config_name)['ProductionVariants'][0]['ModelName']

model_bias_monitor = sagemaker.model_monitor.ModelBiasMonitor(
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sagemaker_session
)

model_bias_monitor.suggest_baseline(
    data_config=sagemaker.clarify.DataConfig(
        s3_data_input_path=f'{p.mb_monitor_dir}/baseline.csv',
        s3_output_path=f'{p.mb_monitor_dir}/info',
        dataset_type = 'text/csv',
        label=target_name,
        predicted_label=prediction_name, 
    ),
    bias_config=sagemaker.clarify.BiasConfig(facet_name='sex_F', label_values_or_threshold=[7], facet_values_or_threshold=[0.5]),
    model_config=sagemaker.clarify.ModelConfig(
        model_name=model_name,
        instance_type='ml.m5.large',
        instance_count=1,
        accept_type='text/csv',
        content_type='text/csv'
    ),
    # model_predicted_label_config=sagemaker.clarify.ModelPredictedLabelConfig(
    #     probability_threshold=0.5  # threshold to convert float prediction to binary label
    # ), 
    wait=True,
    logs=False
)

INFO:sagemaker.image_uris:Defaulting to the only supported framework/algorithm version: 1.0.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.image_uris:Defaulting to the only supported framework/algorithm version: 1.0.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.clarify:Analysis Config: {'dataset_type': 'text/csv', 'label': 'rings', 'predicted_label': 'rings_prediction', 'label_values_or_threshold': [7], 'facet': [{'name_or_index': 'sex_F', 'value_or_threshold': [0.5]}], 'methods': {'report': {'name': 'report', 'title': 'Analysis Report'}, 'pre_training_bias': {'methods': 'all'}, 'post_training_bias': {'methods': 'all'}}, 'predictor': {'model_name': 'abalone-1', 'instance_type': 'ml.m5.large', 'initial_instance_count': 1, 'accept_type': 'text/csv', 'content_type': 'text/csv'}}
INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-05-21-18-49-35-779


...........................................................!

In [45]:
model_bias_monitor.create_monitoring_schedule(
    monitor_schedule_name='abalone-model-bias-monitor',
    endpoint_input=sagemaker.model_monitor.EndpointInput(
        endpoint_name=endpoint_name,
        destination='/opt/ml/processing/input/endpoint',
        inference_attribute='0'
    ),
    ground_truth_input=p.ground_truth_dir,
    output_s3_uri=f'{p.mb_monitor_dir}/reports',
    schedule_cron_expression=sagemaker.model_monitor.CronExpressionGenerator.hourly()
)

INFO:sagemaker.model_monitor.clarify_model_monitoring:Uploading analysis config to {s3_uri}.
INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: abalone-model-bias-monitor


### Predict

In [23]:
test_X=pd.read_csv(p.test_X_file, index_col=0, header=None)
test_X.index.name = None
batch_in_df = test_X.iloc[0:2]
batch_in_df

,1,2,3,4,5,6,7,8,9,10
4163,0.255,0.19,0.070,0.0815,0.0280,0.016,0.031,0,1,0
1270,0.630,0.48,0.145,1.0115,0.4235,0.237,0.305,0,0,1


In [ ]:
# get batch_in_event_ids and save indexless batch_in like X
batch_in_event_ids = batch_in_df.index.to_series()

#batch_in_file=f"{p.batch_in_dir}/{datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H-%M-%S')}"
batch_in_dir=f's3://omm-test-bucket/models/abalone/data/batch-input'
time_string=datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H-%M-%S')
batch_in_file=f"{batch_in_dir}/{time_string}_X.csv"
event_in_file=f"{batch_in_dir}/{time_string}_event_ids.csv"
predictions_file=f"{p.batch_out_dir}/{time_string}_X.csv.out"

batch_in_df.to_csv(batch_in_file, index=False, header=False)
batch_in_df.index.to_series().to_csv(event_in_file, index=False, header=False)

In [29]:
transformer = sagemaker.transformer.Transformer(
    model_name=model_name,
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=p.batch_out_dir+'/',
    accept='text/csv',
    assemble_with='Line',
    sagemaker_session=sagemaker_session
)
transformer.transform(
    data=batch_in_file,
    content_type='text/csv',
    split_type='Line',
    wait=True,
    logs=False
)

# Join results with event IDs after
predictions = pd.read_csv(predictions_file, header=None, names=['prediction'])


INFO:sagemaker:Creating transform job with name: sagemaker-xgboost-2026-05-21-21-32-19-737


.................................................................!


In [ ]:
results = pd.DataFrame({
    'event_id': event_ids,
    'prediction': predictions['prediction']
})

In [46]:
# JSON input
test_X=pd.read_csv(p.test_X_file, index_col=0, header=None)
test_X.index.name = None
test_X.columns = feature_columns
test_dicts=test_X.reset_index().rename(columns={'index': 'id'}).to_dict(orient='records')
model_dict_input=[{k: v for k, v in d.items() if k != target_name} for d in test_dicts]
model_dict_input=[{k: v for k, v in d.items() if k != target_name} for d in test_dicts]

records_to_predict = model_dict_input[0:2]
print(model_dict_input)


[{'id': 4163, 'length': 0.255, 'diameter': 0.19, 'height': 0.07, 'whole_weight': 0.0815, 'shucked_weight': 0.028, 'viscera_weight': 0.016, 'shell_weight': 0.031, 'sex_I': 0, 'sex_M': 1, 'sex_F': 0}, {'id': 1270, 'length': 0.63, 'diameter': 0.48, 'height': 0.145, 'whole_weight': 1.0115, 'shucked_weight': 0.4235, 'viscera_weight': 0.237, 'shell_weight': 0.305, 'sex_I': 0, 'sex_M': 0, 'sex_F': 1}, {'id': 3274, 'length': 0.5, 'diameter': 0.385, 'height': 0.145, 'whole_weight': 0.7615, 'shucked_weight': 0.246, 'viscera_weight': 0.195, 'shell_weight': 0.204, 'sex_I': 0, 'sex_M': 0, 'sex_F': 1}, {'id': 3897, 'length': 0.41, 'diameter': 0.325, 'height': 0.105, 'whole_weight': 0.361, 'shucked_weight': 0.1605, 'viscera_weight': 0.0665, 'shell_weight': 0.103, 'sex_I': 0, 'sex_M': 1, 'sex_F': 0}, {'id': 3864, 'length': 0.695, 'diameter': 0.535, 'height': 0.2, 'whole_weight': 1.5855, 'shucked_weight': 0.667, 'viscera_weight': 0.334, 'shell_weight': 0.471, 'sex_I': 1, 'sex_M': 0, 'sex_F': 0}, {'id':

In [47]:
def predict_request(boto_session, records, featue_columns):
    sm_runtime = boto_session.client('sagemaker-runtime')

    for r in records:
        # row to csv
        row_values=[]
        for column in featue_columns:
            row_values.append(str(r[column]))
        row_csv=','.join(row_values)

        response = sm_runtime.invoke_endpoint(EndpointName=endpoint_name, ContentType='text/csv', Body=row_csv.encode('utf-8'), InferenceId=str(r["id"]))
        print(response)
predict_request(boto_session, records_to_predict, feature_columns)

{'ResponseMetadata': {'RequestId': 'b95ef36e-401b-42a9-9b15-bbebe90003b2', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': 'b95ef36e-401b-42a9-9b15-bbebe90003b2', 'x-amzn-invoked-production-variant': 'AllTraffic', 'date': 'Thu, 21 May 2026 19:00:37 GMT', 'content-type': 'text/csv; charset=utf-8', 'content-length': '18', 'connection': 'keep-alive'}, 'RetryAttempts': 0}, 'ContentType': 'text/csv; charset=utf-8', 'InvokedProductionVariant': 'AllTraffic', 'Body': <botocore.response.StreamingBody object at 0x7fb527977d30>}
{'ResponseMetadata': {'RequestId': '212e28f5-662c-45b3-8dc4-8194f29f6d7c', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '212e28f5-662c-45b3-8dc4-8194f29f6d7c', 'x-amzn-invoked-production-variant': 'AllTraffic', 'date': 'Thu, 21 May 2026 19:00:37 GMT', 'content-type': 'text/csv; charset=utf-8', 'content-length': '19', 'connection': 'keep-alive'}, 'RetryAttempts': 0}, 'ContentType': 'text/csv; charset=utf-8', 'InvokedProductionVariant': 'AllTraffic'

## Ground Truth

In [ ]:
test_y=pd.read_csv(p.test_y_file, index_col=0, header=None)
test_y.index.name = None
test_y.columns=[target_name]
test_dicts=test_y.reset_index().rename(columns={'index': 'id'}).to_dict(orient='records')
ground_truth=[{"value":d[target_name], "event_id":d["id"]} for d in test_dicts]

ground_truths_to_submit = ground_truth[0:2]
print(ground_truth)

# s3://omm-test-bucket/models/abalone/data/capture/abalone-endpoint/AllTraffic/2026/05/21/19/00-37-322-08eeba9c-04c8-4327-8238-d1511d5962a5.jsonl
# {
#   "captureData": {
#     "endpointInput": {
#       "observedContentType": "text/csv",
#       "mode": "INPUT",
#       "data": "MC4yNTUsMC4xOSwwLjA3LDAuMDgxNSwwLjAyOCwwLjAxNiwwLjAzMSwwLDEsMA==",
#       "encoding": "BASE64"
#     },
#     "endpointOutput": {
#       "observedContentType": "text/csv; charset=utf-8",
#       "mode": "OUTPUT",
#       "data": "NS45MDg2NjQyMjY1MzE5ODIK",
#       "encoding": "BASE64"
#     }
#   },
#   "eventMetadata": {
#     "eventId": "b95ef36e-401b-42a9-9b15-bbebe90003b2",
#     "inferenceId": "4163",
#     "inferenceTime": "2026-05-21T19:00:37Z"
#   },
#   "eventVersion": "0"
# }
# {
#   "captureData": {
#     "endpointInput": {
#       "observedContentType": "text/csv",
#       "mode": "INPUT",
#       "data": "MC42MywwLjQ4LDAuMTQ1LDEuMDExNSwwLjQyMzUsMC4yMzcsMC4zMDUsMCwwLDE=",
#       "encoding": "BASE64"
#     },
#     "endpointOutput": {
#       "observedContentType": "text/csv; charset=utf-8",
#       "mode": "OUTPUT",
#       "data": "MTEuMjg3MjAyODM1MDgzMDA4Cg==",
#       "encoding": "BASE64"
#     }
#   },
#   "eventMetadata": {
#     "eventId": "212e28f5-662c-45b3-8dc4-8194f29f6d7c",
#     "inferenceId": "1270",
#     "inferenceTime": "2026-05-21T19:00:37Z"
#   },
#   "eventVersion": "0"
# }


[{'value': 5, 'event_id': 4163}, {'value': 12, 'event_id': 1270}, {'value': 14, 'event_id': 3274}, {'value': 8, 'event_id': 3897}, {'value': 11, 'event_id': 3864}, {'value': 7, 'event_id': 1694}, {'value': 9, 'event_id': 4337}, {'value': 12, 'event_id': 4677}, {'value': 16, 'event_id': 4689}, {'value': 8, 'event_id': 843}, {'value': 9, 'event_id': 4902}, {'value': 7, 'event_id': 2042}, {'value': 7, 'event_id': 3346}, {'value': 10, 'event_id': 807}, {'value': 10, 'event_id': 3821}, {'value': 9, 'event_id': 1918}, {'value': 12, 'event_id': 2754}, {'value': 19, 'event_id': 1151}, {'value': 7, 'event_id': 831}, {'value': 13, 'event_id': 422}, {'value': 13, 'event_id': 491}, {'value': 7, 'event_id': 3464}, {'value': 12, 'event_id': 625}, {'value': 10, 'event_id': 1945}, {'value': 13, 'event_id': 3102}, {'value': 10, 'event_id': 2478}, {'value': 6, 'event_id': 3074}, {'value': 13, 'event_id': 3132}, {'value': 12, 'event_id': 4125}, {'value': 10, 'event_id': 3196}, {'value': 13, 'event_id': 3

In [ ]:
utils.write_ground_truth(boto_session, ground_truths_to_submit, p.ground_truth_dir, created_at=datetime.datetime.now(datetime.timezone.utc))

# s3://omm-test-bucket/models/abalone/data/ground-truth/2026/05/21/19/03-55-055216.jsonl
# {
#   "groundTruthData": {
#     "data": "5",
#     "encoding": "CSV"
#   },
#   "eventMetadata": {
#     "eventId": 4163
#   },
#   "eventVersion": "0"
# }
# {
#   "groundTruthData": {
#     "data": "12",
#     "encoding": "CSV"
#   },
#   "eventMetadata": {
#     "eventId": 1270
#   },
#   "eventVersion": "0"
# }